In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF document
file_path = "employee-handbook.pdf"
loader = PyPDFLoader(file_path)

# Load all pages
docs = loader.load()

print(f"Loaded {len(docs)} pages from the PDF")
print(f"\nFirst page preview (first 500 characters):")
print(docs[0].page_content[:500])
print(f"\nMetadata: {docs[0].metadata}")

Loaded 35 pages from the PDF

First page preview (first 500 characters):
MODEL EMPLOYEE HANDBOOK 
FOR SMALL BUSINESS

Metadata: {'producer': 'QuarkXPress(tm) 6.5', 'creator': 'QuarkXPress(tm) 6.5', 'creationdate': '2005-11-16T16:29:38+00:00', 'moddate': '2017-03-07T10:21:06-05:00', 'xpressprivate': '%%DocumentProcessColors: Cyan Magenta Yellow Black\n%%DocumentCustomColors: (PANTONE 185 C)\n%%+ (PANTONE Blue 072 C)\n%%+ (PANTONE Warm Gray 9 U)\n%%+ (PANTONE 1797 U)\n%%CMYKCustomColor: 0 .91 .76 0 (PANTONE 185 C)\n%%+ 1 .88 0 .05 (PANTONE Blue 072 C)\n%%+ 0 .11 .2 .47 (PANTONE Warm Gray 9 U)\n%%+ 0 1 .99 .04 (PANTONE 1797 U)\n%%EndComments', 'source': 'employee-handbook.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}


In [7]:
# Informational example only – fixed-size chunking WITH overlap
def fixed_size_chunk(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # step back so boundaries overlap
    return chunks

sample = "Employee vacation policy allows 15 days per year. Sick leave is separate..."
print(fixed_size_chunk(sample, chunk_size=40, overlap=10))

['Employee vacation policy allows 15 days ', 's 15 days per year. Sick leave is separa', ' is separate...']


In [8]:
# Informational example only – separator hierarchy (concept behind RecursiveCharacterTextSplitter)
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

def recursive_split(text: str, chunk_size: int, separators: list[str]) -> list[str]:
    if not separators or len(text) <= chunk_size:
        return [text] if text else []
    sep = separators[0]
    parts = text.split(sep) if sep else list(text)
    chunks, current = [], ""
    for part in parts:
        candidate = (current + sep + part) if current else part
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current)
            if len(part) > chunk_size:
                chunks.extend(recursive_split(part, chunk_size, separators[1:]))
                current = ""
            else:
                current = part
    if current:
        chunks.append(current)
    return chunks

In [9]:
# Informational example only – LangChain Markdown header splitter
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
# md_docs = markdown_splitter.split_text(markdown_document)

In [10]:
# Informational example only – LangChain semantic chunker (requires embeddings)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
# semantic_splitter = SemanticChunker(
#     embed_model,
#     breakpoint_threshold_type="percentile",  # split at similarity drops
#     breakpoint_threshold_amount=90,
# )
# semantic_chunks = semantic_splitter.split_text(long_document)

C:\Users\localadmin\AppData\Local\Temp\ipykernel_15088\3049858367.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # Maximum chunk size
    chunk_overlap=200,      # Overlap to maintain context
    add_start_index=True,   # Track position in original doc
    length_function=len,
)

# Split documents into chunks
all_splits = text_splitter.split_documents(docs)

print(f"Split {len(docs)} pages into {len(all_splits)} chunks")
print(f"\nExample chunk:")
print(all_splits[0].page_content)
print(f"\nChunk metadata: {all_splits[0].metadata}")

Split 35 pages into 81 chunks

Example chunk:
MODEL EMPLOYEE HANDBOOK 
FOR SMALL BUSINESS

Chunk metadata: {'producer': 'QuarkXPress(tm) 6.5', 'creator': 'QuarkXPress(tm) 6.5', 'creationdate': '2005-11-16T16:29:38+00:00', 'moddate': '2017-03-07T10:21:06-05:00', 'xpressprivate': '%%DocumentProcessColors: Cyan Magenta Yellow Black\n%%DocumentCustomColors: (PANTONE 185 C)\n%%+ (PANTONE Blue 072 C)\n%%+ (PANTONE Warm Gray 9 U)\n%%+ (PANTONE 1797 U)\n%%CMYKCustomColor: 0 .91 .76 0 (PANTONE 185 C)\n%%+ 1 .88 0 .05 (PANTONE Blue 072 C)\n%%+ 0 .11 .2 .47 (PANTONE Warm Gray 9 U)\n%%+ 0 1 .99 .04 (PANTONE 1797 U)\n%%EndComments', 'source': 'employee-handbook.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1', 'start_index': 0}


In [12]:
from langchain_openai import OpenAIEmbeddings

# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Let's create embeddings for sample texts to understand how they work
sample_texts = [
    "Employee vacation policy and time off",
    "Annual leave and holiday guidelines",
    "Company dress code requirements"
]

# Generate embeddings
sample_embeddings = [embeddings.embed_query(text) for text in sample_texts]

print(f"Each embedding has {len(sample_embeddings[0])} dimensions")
print(f"\nFirst 10 values of embedding 1: {sample_embeddings[0][:10]}")

# Calculate similarity between embeddings (using dot product as approximation)
import numpy as np

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

sim_1_2 = cosine_similarity(sample_embeddings[0], sample_embeddings[1])
sim_1_3 = cosine_similarity(sample_embeddings[0], sample_embeddings[2])

print(f"\nSimilarity between text 1 and 2 (both about time off): {sim_1_2:.4f}")
print(f"Similarity between text 1 and 3 (different topics): {sim_1_3:.4f}")
print("\n✅ Higher similarity = more related content!")

Each embedding has 1536 dimensions

First 10 values of embedding 1: [-0.03509521484375, 0.0113525390625, 0.01983642578125, 0.00884246826171875, -0.0023441314697265625, 0.0214080810546875, -0.0084075927734375, 0.03662109375, 0.03350830078125, 0.00537872314453125]

Similarity between text 1 and 2 (both about time off): 0.5643
Similarity between text 1 and 3 (different topics): 0.2859

✅ Higher similarity = more related content!


In [15]:
from langchain_community.vectorstores import Chroma

# Create Chroma vector store and add documents
# This will:
# 1. Generate embeddings for all chunks
# 2. Store them in Chroma
# 3. Enable similarity search

vector_store = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    persist_directory="./chroma_db"  # Save to disk for reuse
)

print(f"✅ Created vector store with {len(all_splits)} document chunks")
print(f"📁 Data persisted to: ./chroma_db")

✅ Created vector store with 81 document chunks
📁 Data persisted to: ./chroma_db


In [16]:
import chromadb
from langchain_community.vectorstores import Chroma

# Connect to the standalone Chroma server
chroma_client = chromadb.HttpClient(host="localhost", port=8000)

# Quick health check – should print a small integer (nanoseconds since epoch)
print(f"Chroma heartbeat: {chroma_client.heartbeat()}")
print(f"Existing collections: {chroma_client.list_collections()}")

# Create (or open) a collection and ingest the handbook chunks
vector_store = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    client=chroma_client,
    collection_name="employee_handbook",
)

print(f"✅ Connected to standalone Chroma at localhost:8000")
print(f"✅ Collection 'employee_handbook' now has {len(all_splits)} chunks")

Chroma heartbeat: 1787728709436686100
Existing collections: []
✅ Connected to standalone Chroma at localhost:8000
✅ Collection 'employee_handbook' now has 81 chunks
